In [1]:
"""
Изучить базовые операции PySpark DataFrame API.

Научиться профилировать использование CPU (ядер) и памяти при обработке данных.

Сравнить производительность и потребление ресурсов между Pandas (однопоточный, in‑memory) и PySpark (распределённый, ленивые вычисления).

Понять, когда и почему Spark выигрывает или проигрывает.
"""

_IncompleteInputError: incomplete input (969763094.py, line 1)

In [ ]:
"""
Лабораторная работа: Сравнение Pandas и PySpark с профилированием ресурсов.
вывод результатов. Всю логику обработки пишете сами.
"""

import time
import numpy as np
import pandas as pd
import psutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================
# 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ (ДАНО)
# ============================

def get_memory_usage_pandas():
    """Использование памяти процессом Python (МБ)"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_spark_resources(spark):
    """Возвращает (число ядер, партиций по умолчанию, память экзекьюторов в МБ)"""
    cores = spark.sparkContext.defaultParallelism
    partitions = spark.sparkContext.defaultParallelism
    memory_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
    total_memory_mb = sum(entry._2() for entry in memory_status.values()) / (1024 * 1024)
    return cores, partitions, total_memory_mb

# ============================
# 2. ГЕНЕРАЦИЯ ДАННЫХ (TODO)
# ============================

def generate_data(num_days, num_patients, seed=42):
    """
    Генерирует два pandas.DataFrame:
      - logs: колонки [timestamp, patient_id, hr_raw, spo2] с 5% пропусков
      - meta: колонки [patient_id, age, diagnosis]
    TODO: 
      1. Создать временные метки с шагом 1 минута на num_days дней.
      2. Для каждого пациента сгенерировать ЧСС (hr_raw) как база + случайное блуждание + шум.
      3. SpO2 = 98 - |нормальное(0,2)|.
      4. Объединить все записи в один DataFrame.
      5. Внести 5% пропусков в hr_raw и spo2 (случайные позиции).
      6. Сгенерировать мета-данные: возраст от 25 до 85, диагноз из ["Healthy","Arrhythmia","Hypertension"].
    Вернуть (logs, meta).
    """
    np.random.seed(seed)
    # TODO: напишите генерацию
    # Подсказка: используйте pd.date_range, цикл по пациентам, списки записей
    logs = ...   # должен быть pandas.DataFrame
    meta = ...
    return logs, meta

# ============================
# 3. PANDAS ПАЙПЛАЙН (TODO: напишите замеры и обработку)
# ============================

def pandas_processing(logs_path, meta_path, label):
    """
    Загружает CSV, выполняет очистку и агрегацию (как в лабе по Pandas).
    TODO:
      - Замерить память ДО загрузки (get_memory_usage_pandas)
      - Замерить время начала
      - Прочитать logs и meta через pd.read_csv
      - Преобразовать timestamp в datetime
      - Сделать merge
      - Линейная интерполяция пропусков (hr_raw, spo2)
      - Группировка по patient_id + скользящее среднее 15 мин (rolling)
      - Определить тахикардию (>100)
      - Создать колонку date из индекса
      - Агрегация: средний пульс, мин SpO2, сумма тахикардии по (patient_id, date)
      - Замерить время окончания и память ПОСЛЕ
    Вернуть (result_df, elapsed_time_sec, memory_delta_mb)
    """
    # TODO: реализовать
    mem_before = ...
    start = ...
    # ... весь пайплайн ...
    elapsed = ...
    mem_after = ...
    mem_delta = mem_after - mem_before
    return daily_stats, elapsed, mem_delta

# ============================
# 4. SPARK ПАЙПЛАЙН (TODO)
# ============================

def spark_processing(logs_path, meta_path, label):
    """
    Загружает CSV в Spark DataFrame, выполняет те же операции, что и в pandas.
    TODO:
      - Создать SparkSession (имя = f"MedicalAnalysis_{label}")
      - Получить ресурсы через get_spark_resources(spark) и вывести на печать
      - Замерить время начала
      - Прочитать logs и meta (spark.read.csv с inferSchema=True)
      - Преобразовать timestamp в TimestampType (to_timestamp)
      - Выполнить LEFT JOIN
      - Заполнить пропуски: forward fill (last(..., ignorenulls=True) over window по patient_id, order by timestamp, rows между unbounded и 0),
        затем backward fill (first(...) over window rows между 0 и unbounded)
      - Скользящее среднее за 15 минут (окно 15 строк: rowsBetween(-14,0))
      - Создать колонку is_tachycardia (hr_smooth > 100)
      - Извлечь дату из timestamp (date_format или to_date)
      - Сгруппировать по patient_id, date и вычислить:
          * среднее hr_smooth
          * минимум spo2
          * сумму is_tachycardia (как целое число)
      - Принудительно запустить вычисления (собрать в pandas или просто вызвать count)
      - Замерить время окончания
      - Вывести количество партиций у финального DataFrame (df.rdd.getNumPartitions())
      - Остановить SparkSession
    Вернуть (result_pandas_df, elapsed_time_sec, final_partitions, cores, exec_memory_mb)
    """
    spark = SparkSession.builder.appName(f"MedicalAnalysis_{label}").getOrCreate()
    cores, default_parts, exec_mem = get_spark_resources(spark)
    print(f"Spark ресурсы: ядер={cores}, партиций по умолч={default_parts}, память экзекьюторов={exec_mem:.0f} МБ")
    
    start = time.time()
    
    # TODO: ваш код Spark (чтение, join, fill, rolling, agg)
    
    elapsed = time.time() - start
    
    # TODO: получить число партиций (например, через df.rdd.getNumPartitions() после агрегации)
    final_partitions = ...
    
    # TODO: собрать результат в pandas (daily_stats.toPandas())
    result_pd = ...
    
    spark.stop()
    return result_pd, elapsed, final_partitions, cores, exec_mem

# ============================
# 5. ЭКСПЕРИМЕНТЫ (TODO)
# ============================

def run_experiment(num_days, num_patients, label):
    """
    Генерирует данные, запускает Pandas и Spark обработку, возвращает словарь с метриками.
    TODO:
      - Вызвать generate_data(num_days, num_patients)
      - Сохранить logs и meta в CSV (без индекса)
      - Вызвать pandas_processing() и spark_processing() с путями к этим CSV
      - Вернуть словарь: {'label': label, 'pandas_time': ..., 'spark_time': ...,
                          'pandas_mem': ..., 'spark_exec_mem': ..., 'spark_cores': ...,
                          'spark_partitions': ...}
    """
    # TODO: реализовать
    logs, meta = ...
    logs.to_csv(...)
    meta.to_csv(...)
    pd_res, pd_time, pd_mem = pandas_processing(...)
    sp_res, sp_time, sp_parts, sp_cores, sp_mem = spark_processing(...)
    return {...}

# ============================
# 6. ГЛАВНЫЙ БЛОК (запуск сравнения)
# ============================

if __name__ == "__main__":
    # Малый эксперимент: 5 дней, 3 пациента
    small = run_experiment(5, 3, "малый (5 дней, 3 пациента)")
    # Большой эксперимент: 30 дней, 100 пациентов
    large = run_experiment(30, 100, "большой (30 дней, 100 пациентов)")
    
    # Сводная таблица
    results = pd.DataFrame([small, large])
    print("\n=== СВОДНЫЕ РЕЗУЛЬТАТЫ ===")
    print(results.to_string(index=False))
    
    # TODO: постройте графики 